# `df.style`

possible API:

We'd like to conditionally format a dataframe. We'd like to be able to format cells based on

- The cell's value
- The cell's value relative to the column
- The cell's position in the table

(these may be incompatible... not sure).

The API should be somthing like

```python
df.style(rows={'A': func1, 'B': func2}, columns=None, table=None)
```

or maybe 

```python
df.style(rows={'A': [func1, func2], 'B': func2}, table=None)
```

or maybe

```python
df.style(columns=func1)
```

The last example is sugar for `df.style(columns=dict(zip(df.columns, cycle([func1])))` i.e.
apply `func1` to each column.
What's the API for each `func`?

```python
def f(values)
    pass
```

where `values` is either a Series for `rows` or `columns`, or a DataFrame for `table`.

**Each style function must return a like-indexed Series/DataFrame of (css-selecter, value) pairs**.

This should cover most cases. The only other thing is *maybe* the index / columns. Optionally allow for `index=` and `header=`. Maybe.

I think this strikes a reasonable balance between complexity / flexibility. Users can "share" styles.

# Background

- https://github.com/pydata/pandas/issues/3190


In [95]:
from matplotlib.colors import rgb2hex
import pandas.core.common as com

In [145]:
def monkeypatch_method(cls):
    def decorator(func):
        setattr(cls, func.__name__, func)
        return func
    return decorator

@monkeypatch_method(pd.DataFrame)
def style(self, rows=None, columns=None, frame=None):
    row_styles, col_styles, frame_styles = self._build_styles(
        rows=rows, columns=columns, frame=frame
    )
    self.to_html(styles)

@monkeypatch_method(pd.DataFrame)
def _format_sytle_args(self, rows=None, columns=None, frame=None):
    if rows is not None:
        # a signle function, apply to each row
        if callable(rows):
            rows = {row: [rows] for row in self.index}
        else:
            rows = {row: [x] if not com.is_list_like(x) else x
                    for row, x in rows.items()}
    if columns is not None:
        if callable(columns):
            columns = {col: [columns] for col in self.columns}
        else:
            columns = {col: [x] if not com.is_list_like(x) else x
                       for col, x in columns.items()}
    if frame is not None:
        frame = [x] if not com.is_list_like(x) else x
    return rows, columns, frame

@monkeypatch_method(pd.DataFrame)
def _build_styles(self, rows=None, columns=None, frame=None):
    final = []  # (row, col, prop, value)

    rows, columns, frame = self._format_sytle_args(rows, columns, frame)

    if rows is not None:
        for row in rows:
            for func in rows[row]:
                props, values  = zip(*func(self.loc[row]))
                final.append(pd.DataFrame({'row': row, 'col': self.columns,
                                           'prop': props, 'value': values}))
    if columns is not None:
        for col in columns:
            for func in columns[col]:
                props, values  = zip(*func(self[col]))
                final.append(pd.DataFrame({'row': self.index, 'col': col,
                                           'prop': props, 'value': values}))

    if frame is not None:
        for func in frame:
            r = func(self)
            for col in r:
                try:
                    prop, values = zip(*r[col])
                except ValueError:
                    prop, values = '', ''
                    final.append(pd.DataFrame({'row': r.index, 'col': col,
                                           'prop': props, 'value': values}))

    r = pd.concat(final)[['row', 'col', 'prop', 'value']]
    r = (r['prop'] + ':' + r['value']).groupby([r['row'], r['col']]).agg(
         lambda x: '; '.join(x))

    return r.unstack()



In [146]:
def color_bg_range(s, cmap='PuBu'):
    """Color background in a range."""
    colors = [rgb2hex(x) for x in plt.cm.get_cmap(cmap)(s)]
    return pd.Series([('backgroud-color', color) for color in colors])

def color_font_even(s):
    colors = ['black' if x % 2 else 'red' for x in s]
    return pd.Series([('font-color', color) for color in colors])

def color_gradient(s, cmap='RdBu'):
    colors = [rgb2hex(x) for x in plt.cm.get_cmap(cmap)(range(len(s)))]
    return pd.Series([('background-color', color) for color in colors])

def striped(s):
    colors = ["B5DEFF" if i % 2 else "" for i in range(len(s))]
    return pd.Series([('background-color', color) for color in colors])

In [147]:
# this is a bit strange. Striping the rows, but applying
# to each column.
df._build_styles(columns=striped)

col,A,B
row,,
0,background-color:,background-color:
1,background-color:B5DEFF,background-color:B5DEFF
2,background-color:,background-color:
3,background-color:B5DEFF,background-color:B5DEFF
4,background-color:,background-color:


In [124]:
np.random.seed(24)
s = pd.DataFrame({'A': np.random.permutation(range(6))})

df = pd.DataFrame({'A': np.linspace(1, 5, 5), 'B': np.random.randn(5)})
df

,A,B
0,1,-0.609561
1,2,-1.228095
2,3,-0.808719
3,4,-0.408644
4,5,-1.548511


In [125]:
df._build_styles(columns={"A": [color_bg_range, color_font_even], 'B': color_bg_range})

col,A,B
row,,
0,backgroud-color:#023858; font-color:black,backgroud-color:#fff7fb
1,backgroud-color:#023858; font-color:red,backgroud-color:#fff7fb
2,backgroud-color:#023858; font-color:black,backgroud-color:#fff7fb
3,backgroud-color:#023858; font-color:red,backgroud-color:#fff7fb
4,backgroud-color:#023858; font-color:black,backgroud-color:#fff7fb


In [62]:
df._build_styles(columns={'A': color_gradient})

col,A
row,
0,background-color:#67001f
1,background-color:#6a011f
2,background-color:#6d0220
3,background-color:#700320
4,background-color:#730421


In [72]:
def is_positive_bg(s):
    colors = {True: ('backgroud-color', 'teal'),
              False: ('background-color', 'blue')}
    if isinstance(s, pd.Series):
        return (s > 0).map(colors)
    else:
        return (s > 0).applymap(lambda x: colors[x])

def is_even_font(s):
    colors = {True: ('font-color', 'red'),
              False: ('font-color', 'yellow')}
    if isinstance(s, pd.Series):
        return (s % 2).map(colors)
    else:
        return(s % 0).applymap(lambda x: colors[x])


In [74]:
pairs = is_positive_bg(df)
pairs

,A,B
0,"(backgroud-color, teal)","(background-color, blue)"
1,"(backgroud-color, teal)","(background-color, blue)"
2,"(backgroud-color, teal)","(background-color, blue)"
3,"(backgroud-color, teal)","(background-color, blue)"
4,"(backgroud-color, teal)","(background-color, blue)"


In [94]:
df.style(columns={'A': is_even_font}, frame=[is_positive_bg])

NameError: name 'x' is not defined

In [25]:
df.apply(maybe_is_positive_bg)

,A,B
0,,background-color: blue
1,,background-color: blue
2,,background-color: blue
3,,
4,,background-color: blue


In [29]:
df.head()

,A,B
0,1,-1.683868
1,2,-0.252138
2,3,-3.282090
3,4,1.783877
4,5,-1.314358


In [28]:
df.apply(maybe_is_positive_bg, axis=0)

,A,B
0,,background-color: blue
1,,background-color: blue
2,,background-color: blue
3,,
4,,background-color: blue


In [39]:
def f(x):
    if x >= 0:
        return 'background-color: blue'
    else:
        return ''

In [8]:
def maybe_is_positive_bg(s):
    colors = {True: '',
              False: 'background-color: blue'}
    if isinstance(s, pd.Series):
        return (s > 0).map(colors)
    else:
        return (s > 0).applymap(lambda x: colors[x])

# y-p's work

In [75]:
from jinja2 import Template
# The baseline jinja2 template for HTML output.
t=Template("""
<style  type="text/css" >
#T_{{uuid}} tr {
border: none;
}
#T_{{uuid}} {
border: none;
}
#T_{{uuid}} th.blank {
border: none;
}


{% for s in style %}
#T_{{uuid}} {{s.selector}} {  
{% for p,val in s.props %}
    {{p}}: {{val}};
{% endfor %}
  }
{% endfor %}
</style>
<table id="T_{{uuid}}">
{% if caption %}
<caption>
{{caption}}
</caption>
{% endif %}

  <thead>
{% for r in head %}
    <tr>
  {% for c in r %}    
      <{{c.type}} class="{{c.class}}">{{c.value}}</th>
  {% endfor %}
    </tr>
{% endfor %}
  </thead>
  <tbody>
 {% for r in body %}
    <tr>
  {% for c in r %}    
      <{{c.type}} class="{{c.class}}">{{c.value}}</th>
  {% endfor %}
    </tr>
{% endfor %}
  </tbody>

</table>

""")


In [76]:
ROW_HEADING_CLASS="row_heading"
COL_HEADING_CLASS="col_heading"
DATA_CLASS="data"
BLANK_CLASS="blank"
BLANK_VALUE=""

def translate(df,cell_context=None):
    import uuid
    cell_context = cell_context or dict()

    n_rlvls =df.index.nlevels
    n_clvls =df.columns.nlevels
    rlabels=df.index.tolist()
    clabels=df.columns.tolist()
    if n_rlvls == 1:
        rlabels = [[x] for x in rlabels]
    if n_clvls == 1:
        clabels = [[x] for x in clabels]
    clabels=list(zip(*clabels))
    head=[]
    for r in range(n_clvls):    
        row_es = [{"type":"th","value":BLANK_VALUE ,"class": " ".join([BLANK_CLASS])}]*n_rlvls
        for c in range(len(clabels[0])):
            cs = [COL_HEADING_CLASS,"level%s" % r,"col%s" %c]
            cs.extend(cell_context.get("col_headings",{}).get(r,{}).get(c,[]))
            row_es.append({"type":"th","value": clabels[r][c],"class": " ".join(cs)})
        head.append(row_es)
    body=[]
    for r in range(len(df)):    
        cs = [ROW_HEADING_CLASS,"level%s" % c,"row%s" % r]
        cs.extend(cell_context.get("row_headings",{}).get(r,{}).get(c,[]))
        row_es = [{"type":"th","value": rlabels[r][c],"class": " ".join(cs)} 
                  for c in range(len(rlabels[r]))]
        for c in range(len(df.columns)):
            cs = [DATA_CLASS,"row%s" % r,"col%s" %c]
            cs.extend(cell_context.get("data",{}).get(r,{}).get(c,[]))
            row_es.append({"type":"td","value": df.iloc[r][c],"class": " ".join(cs)})
        body.append(row_es)

    # uuid required to isolate table styling from others 
    # in same notebook in ipnb
    u = str(uuid.uuid1()).replace("-","_")
    return dict(head=head, body=body,uuid=u)


In [22]:
df=mkdf(10,5,r_idx_nlevels=3,c_idx_nlevels=2)
from IPython.display import HTML,display
ctx= translate(df)
ctx['caption']="Just a table, but rendered using a template with lots of classes to style against"
display(HTML(t.render(**ctx)))


In [37]:
def zebra(color1, color2):
    return [dict(selector="td.data:nth-child(2n)" ,
                props=[("background-color",color1)]),
            dict(selector="td.data:nth-child(2n+1)" ,
                props=[("background-color",color2)])]

ctx= translate(df)
style=[]
style.extend(zebra("#aaa","#ddd"))
ctx['style']=style
ctx['caption']="A zebra table"
display(HTML(t.render(**ctx)))

In [88]:
t.render(**ctx).split('\n')

['',
 '<style  type="text/css" >',
 '#T_fae2e978_065f_11e5_96fa_a45e60bd97fb tr {',
 'border: none;',
 '}',
 '#T_fae2e978_065f_11e5_96fa_a45e60bd97fb {',
 'border: none;',
 '}',
 '#T_fae2e978_065f_11e5_96fa_a45e60bd97fb th.blank {',
 'border: none;',
 '}',
 '',
 '',
 '',
 '#T_fae2e978_065f_11e5_96fa_a45e60bd97fb td.data:nth-child(2n) {  ',
 '',
 '    background-color: #aaa;',
 '',
 '  }',
 '',
 '#T_fae2e978_065f_11e5_96fa_a45e60bd97fb td.data:nth-child(2n+1) {  ',
 '',
 '    background-color: #ddd;',
 '',
 '  }',
 '',
 '</style>',
 '<table id="T_fae2e978_065f_11e5_96fa_a45e60bd97fb">',
 '',
 '<caption>',
 'A zebra table',
 '</caption>',
 '',
 '',
 '  <thead>',
 '',
 '    <tr>',
 '      ',
 '      <th class="blank"></th>',
 '      ',
 '      <th class="blank"></th>',
 '      ',
 '      <th class="blank"></th>',
 '      ',
 '      <th class="col_heading level0 col0">C_l0_g0</th>',
 '      ',
 '      <th class="col_heading level0 col1">C_l0_g1</th>',
 '      ',
 '      <th class="col_head